In [2]:

#importing the file from github and storing it
import urllib.request

url=("https://raw.githubusercontent.com/rasbt/"
"LLMS-from-scratch/main/ch02/01_main-chapter-code/"
"the-verdict.txt")

file_path="the-verdict.txt"
urllib.request.urlretrieve(url,file_path)


('the-verdict.txt', <http.client.HTTPMessage at 0x1fa2d74c860>)

In [3]:
#opening the file and reading it
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text=f.read()

    print("total number of characters:",len(raw_text))
    print(raw_text[:99])

total number of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [4]:
#now splitting the text into words
import re

#test = "Hello, world. Is this-- a test?"
#result= re.split(r'([,.;:?_!"()\']|\s|--|\d)',test)
#result=[item for item in result if item.strip()]
#print(result)

#the story
preprocessed=re.split(r'([,.;:?_!"()\']|\s|--|\d)',raw_text)
preprocessed=[item for item in preprocessed if item.strip()]
print(len(preprocessed))
print(preprocessed[4680:4690])

# now we have a list of all individual words and special tokens, now we need to create a vocab and map it to integers and vice versa





4690
["'", 's', 'no', 'exterminating', 'our', 'kind', 'of', 'art', '.', '"']


In [5]:
#creating vocab and mapping it to integers and vice versa
WordsinVocab=sorted(set(preprocessed))
WordsinVocab.extend(["<|endoftext|>","<unk>"])
vocab_size=len(WordsinVocab)

print("vocab size:",vocab_size,"last 5:",WordsinVocab[-5:])

# now the mapping of words to integers and vice versa for the vocab
vocab={token:i for i,token in enumerate(WordsinVocab)}

for i,item in enumerate(list(vocab.items())[-5:]):
    print(i,item)

  

vocab size: 1132 last 5: ['younger', 'your', 'yourself', '<|endoftext|>', '<unk>']
0 ('younger', 1127)
1 ('your', 1128)
2 ('yourself', 1129)
3 ('<|endoftext|>', 1130)
4 ('<unk>', 1131)


In [6]:
#now the whole tokenzier class with encode and decode methods

class TokenizerV1:
    def __init__(self, vocab):
        self.str_to_int=vocab
        self.int_to_str={i:token for token,i in vocab.items()}

    def encode(self,text):
        preprocessed=re.split(r'([,.;:?_!"()\']|\s|--|\d)',text)
        preprocessed=[item for item in preprocessed if item.strip()]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self,ids):
        text=" ".join([self.int_to_str[i]for i in ids])
        text= re.sub(r'\s+([,.?!:;"()\'])',r'\1',text)
        return text


In [7]:
#testing the tokenizer class

import torch
import numpy
tokenizer=TokenizerV1(vocab)
text= """" It's the last he painted, you know," 
       Mrs. Gisburn said with pardonable pride."""
ids=tokenizer.encode(text)
print(ids)


dec=tokenizer.decode(ids)
print(dec)



[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [8]:
# new tokenizer class with <unk> and <|endoftext|> tokens

class TokenizerV2:
    def __init__(self, vocab):
        self.str_to_int=vocab
        self.int_to_str={index:token for token,index in vocab.items()}

    def encode(self,text):
        preprocessed=re.split(r'([,.;:?_!"()\']|\s|--|\d)',text)
        preprocessed=[item for item in preprocessed if item.strip()]
        preprocessed=[item if item in self.str_to_int 
                      else "<unk>" for item in preprocessed]
        
        ids=[self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self,ids):
        text=" ".join([self.int_to_str[i] for i in ids])
        text=re.sub(r'\s+([,.?!:;"()\'])',r'\1',text)
        return text

In [9]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text=" <|endoftext|> ".join([text1,text2])
print(text)

#checking the tokenizerv2
tokenizer2= TokenizerV2(vocab)
print(tokenizer2.encode(text))
print(tokenizer2.decode(tokenizer2.encode(text)))

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.
[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
<unk>, do you like tea? <|endoftext|> In the sunlit terraces of the <unk>.


# Day 1 last
# Tiktokenizer and BPE

In [10]:
# Now Byte pair encoding
#using tiktoken

import tiktoken
# version check for tiktoken
from importlib.metadata import version
print("tiktoken version:", version("tiktoken"))

# now actual byte pair encoding using tiktoken
TTokenizer=tiktoken.get_encoding("gpt2")

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace. Or maybe someunknownplace"
text=" <|endoftext|> ".join([text1,text2])

integer=TTokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integer)

#now we decode
string=TTokenizer.decode(integer)
print(string)

#unknown test
test= "Akwirw ier"

result=TTokenizer.encode(test)
print(result)

reverse=TTokenizer.decode(result)
print(reverse)

tiktoken version: 0.13.0
[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 262, 20562, 13, 1471, 3863, 617, 34680, 5372]
Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace. Or maybe someunknownplace
[33901, 86, 343, 86, 220, 959]
Akwirw ier


# NOW we are doing the BPE tokenizer and making input and target pairs using tiktokenizer

In [11]:
# DAY 2 (coding days)

# reading file already done in previous sections
# now encoding the whole text using tiktoken

enc_text = TTokenizer.encode(raw_text)
print(len(enc_text))

# removing a sample of tokens
enc_sample = enc_text[50:]
print(len(enc_sample))

5145
5095


In [12]:
# NOW input output pairs for training the model, x has input tokens and y has the targets,
#which is just input shifted by 1

#Context size determines how many tokens are included in the input
context_size=4
# 0-3
x= enc_sample[:context_size]
# 1-4
y=enc_sample[1:context_size+1]

print(f"x:{x}")
print(f"y:{y}")


x:[290, 4920, 2241, 287]
y:[4920, 2241, 287, 257]


In [13]:
# Now the next word prediction
for i in range(1,context_size+1):
    context= enc_sample[:i]
    desired=enc_sample[i]
    #print(context, "------>" , desired)

    #printing the actual words for the input and output tokens
    print(TTokenizer.decode(context), "------>" , TTokenizer.decode([desired]))


 and ------>  established
 and established ------>  himself
 and established himself ------>  in
 and established himself in ------>  a


In [14]:
# Dataset and Dataloader-> Dataloader will return x and y in batches, so we can train the model on those batches

# using the pytorch dataset and dataloader

import torch
from torch.utils.data import Dataset, DataLoader


#making a dataset class

class LLMDatasetV1(Dataset):
#text is the whole text, tokenizer is the tiktoken tokenizer, max_length is the context size, stride is how many tokens to move forward for the next input
    def __init__(self,text,tokenizer,max_length,stride):
        self.input_ids=[]
        self.target_ids=[]

        token_ids=TTokenizer.encode(text)

        for i in range(0,len(token_ids)-max_length,stride):
            input_chunk= token_ids[i:i+max_length]
            target_chunks=token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))  
            self.target_ids.append(torch.tensor(target_chunks))

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]
    



# Now the dataloader function 

def create_DataloaderV1(text, batch_size=4, max_length=256, stride=128,shuffle=True, drop_last=True, num_workers=0):
    
    dataset= LLMDatasetV1(text,TTokenizer,max_length,stride)
    dataLoader= DataLoader(dataset,batch_size=batch_size,shuffle=shuffle,
                           drop_last=drop_last,num_workers=num_workers)
    return dataLoader



In [15]:
# Testing the dataset and dataloader

#raw_text already defined above

dataloader= create_DataloaderV1(raw_text,batch_size=1,max_length=4,stride=4,shuffle=False)
data_iter=iter(dataloader)
first_batch=next(data_iter)
print(first_batch)

#second batch
second_batch = next(data_iter)
#print(second_batch[0])



[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [16]:
#data 
dataloader= create_DataloaderV1(raw_text,batch_size=8,max_length=4,stride=4,shuffle=False)
data_iter=iter(dataloader)
inputs,targets = next(data_iter)
print("Inputs \n",inputs)
print("Targets \n",targets)

Inputs 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets 
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


# DAY 3 (coding days)
# Now we will make this into embeddings vectors


In [20]:
# Token ids to embedding vector
# we must assign random weight values and GPT like LLMS work on backpropgation (neural nets, calculus--> Thanks Andrej)

#example conversion
inputID= torch.tensor([2,3,5,1])
# suppose vocab is 6 and we are creating embeddings of 3 dimension (GPT 3  had 12.288k)
vocabsize=6
outputdim=3
#setting random seed to 123 (to randomize weights in a reproductible way)
torch.manual_seed(123)
embedding=torch.nn.Embedding(vocabsize,outputdim)
print("embedding weight: ", embedding.weight)
print("embedding torch.tensor : ",embedding(torch.tensor([3])))
print("embedding inputID: ",embedding(inputID))

embedding weight:  Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)
embedding torch.tensor :  tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)
embedding inputID:  tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [23]:
# now for our current progress so far

print(vocab_size)
# we are using tiktoken'z gpt2 tokenizer and its vocab is 50257 words
Tvocab_size=50257
output_dim=256
token_embedding_layer=torch.nn.Embedding(Tvocab_size,output_dim)
# now we see the batch previously implemented
print("Inputs \n",inputs)
print("Inputs Shape \n",inputs.shape)

# Now the embeddings
token_embeddings=token_embedding_layer(inputs)
print("token_embeddings.shape: ", token_embeddings.shape)
print("token_embeddings: ",token_embeddings)




1132
Inputs 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Inputs Shape 
 torch.Size([8, 4])
token_embeddings.shape:  torch.Size([8, 4, 256])
token_embeddings:  tensor([[[ 1.3867,  0.3648, -1.6636,  ...,  0.9439, -0.1907, -0.5579],
         [-1.9067, -1.1811,  0.6944,  ...,  0.6617,  2.2738, -1.1927],
         [-1.4966,  0.4130,  0.8030,  ..., -0.6872, -0.8331,  0.7569],
         [ 0.8888, -0.8464, -1.2117,  ...,  1.6264,  0.1823, -0.2117]],

        [[ 0.2282, -1.4490,  0.9381,  ...,  1.6701,  0.4581, -1.4488],
         [ 0.3890, -0.2291,  1.0056,  ...,  1.0879,  0.6388, -0.3441],
         [-0.7427, -0.3442,  0.4775,  ...,  1.6191,  2.1775, -2.1619],
         [-0.3239,  1.5374, -1.7144,  ...,  0.3760,  0.7733,  1.7882]],

       

In [24]:
# now the positonal embeddings, we need another embedding layer
#max_length is 4
max_length =4 
context_length= max_length
pos_embedding_layer=torch.nn.Embedding(context_length,output_dim)
pos_embeddings=pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

#add it to token embeddings 
input_embeddings= token_embeddings + pos_embeddings
print(input_embeddings.shape)



torch.Size([4, 256])
torch.Size([8, 4, 256])
